# ITELv5 - SECCIÓN 8: EXPORTACIÓN DEL MODELO FINAL

En esta sección, usamos los mejores hiperparámetros encontrados en la Sección 6 para entrenar un modelo MLP final y un Scaler final sobre *todo* el conjunto de datos (3,730 audios).

Estos archivos (`final_mlp_model.pt` y `final_mlp_scaler.joblib`) son los "artefactos" de producción que se usarán para la inferencia en un servidor o API.

In [1]:
# 8.5.1 - Instalar ONNX
import sys
!{sys.executable} -m pip install onnx onnxruntime

print("\n✅ Paquetes 'onnx' y 'onnxruntime' instalados.")


✅ Paquetes 'onnx' y 'onnxruntime' instalados.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 8.1 - Configuración e importaciones
%run "./ITELv5_01_CONFIGURACION.ipynb"

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import joblib # Usaremos joblib para guardar el scaler
from sklearn.preprocessing import StandardScaler

# Asegurarse de que las métricas (usadas en HPO) están importadas si se corre 06
try:
    from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score
except ImportError:
    pass

print("✅ Entorno de exportación listo.")

🔍 Verificando estructura de carpetas...

✅ Estructura de carpetas completa.
🔧 Dependencias comprobadas.

📁 BASE_DIR: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL
🖥️ GPU disponible: False
Rutas Principales de ITELv5:
Datos_Brutos:     C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Datos_Brutos
Datos Procesados: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Datos_Procesados
Modelos:          C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos
Reportes (CSV):   C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV
Reportes (Figuras): C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\Figuras
Logs:             C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Logs
🔍 Carpeta Healthy con 308 subrutas de vocales
🔍 Carpeta Patients con 378 subrutas de vocales
✅ Configuración ITELv5 lista.
✅ Entorno de exportación listo.


In [3]:
# 8.2 - Definir la Arquitectura del Modelo MLP
# (Copiado de 06_Training)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(nn.Module):
    def __init__(self, in_dim=4608, hidden=512, dropout=0.3, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, num_classes)
        )
    def forward(self, x):
        return self.net(x)

print(f"✅ Definición MLP lista. Usando device: {device}")

✅ Definición MLP lista. Usando device: cpu


In [4]:
# 8.3 - Cargar Datos Completos y Mejores HParams

print("Cargando embeddings (X, y) y mejores HParams...")

# 1. Cargar Embeddings y Info (de 05)
emb_dir = ITELConfig.PROCESADOS_DIR 
npy_file = emb_dir / 'embeddings_v5.npy'
info_file = emb_dir / 'embeddings_info_v5.csv'

if not npy_file.exists() or not info_file.exists():
    raise FileNotFoundError(f'No se encontraron embeddings en {emb_dir}. Ejecuta la Sección 5.')

X_full = np.load(npy_file)
info = pd.read_csv(info_file)

# 2. Mapear Labels
label_map = {'Healthy': 0, 'Patients': 1}
y_full = info['categoria'].map(label_map).values

print(f"Datos cargados: X shape {X_full.shape}, y shape {y_full.shape}")

# 3. Cargar HParams (de 06)
hparams_path = ITELConfig.MODELS_DIR / 'mlp_hparams_v5.json'
try:
    with open(hparams_path, 'r') as f:
        hparams = json.load(f)
    print(f"Mejores HParams cargados: {hparams}")
except FileNotFoundError:
    print(f"⚠️ No se encontró {hparams_path}. Usando defaults. (Re-ejecutar 06 si es necesario)")
    hparams = {'hidden': 256, 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 1e-05, 'epochs': 100}

Cargando embeddings (X, y) y mejores HParams...
Datos cargados: X shape (3730, 4608), y shape (3730,)
Mejores HParams cargados: {'dropout': 0.5, 'epochs': 100, 'hidden': 256, 'lr': 0.001, 'weight_decay': 1e-05}


In [5]:
# 8.4 - Entrenar y Guardar el Scaler Final

print("Entrenando el Scaler final sobre TODOS los datos...")

# 1. Instanciar y entrenar el scaler
final_scaler = StandardScaler()
X_scaled_full = final_scaler.fit_transform(X_full)

# 2. Guardar el scaler usando joblib
scaler_path = ITELConfig.MODELS_DIR / 'final_mlp_scaler.joblib'
joblib.dump(final_scaler, scaler_path)

print(f"✅ Scaler final entrenado y guardado en: {scaler_path}")

Entrenando el Scaler final sobre TODOS los datos...
✅ Scaler final entrenado y guardado en: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos\final_mlp_scaler.joblib


In [6]:
# 8.4 - Entrenar y Guardar el Scaler Final

print("Entrenando el Scaler final sobre TODOS los datos...")

# 1. Instanciar y entrenar el scaler
final_scaler = StandardScaler()
X_scaled_full = final_scaler.fit_transform(X_full)

# 2. Guardar el scaler usando joblib
scaler_path = ITELConfig.MODELS_DIR / 'final_mlp_scaler.joblib'
joblib.dump(final_scaler, scaler_path)

print(f"✅ Scaler final entrenado y guardado en: {scaler_path}")

Entrenando el Scaler final sobre TODOS los datos...
✅ Scaler final entrenado y guardado en: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos\final_mlp_scaler.joblib


In [7]:
# 8.5 - Entrenar y Guardar el Modelo MLP Final

print("Entrenando el Modelo MLP final sobre TODOS los datos...")

# 1. Convertir datos escalados a Tensores
X_scaled_t = torch.tensor(X_scaled_full, dtype=torch.float32).to(device)
y_full_t = torch.tensor(y_full, dtype=torch.long).to(device)

# 2. Instanciar modelo y optimizador
final_model = MLP(
    in_dim=X_scaled_full.shape[1], 
    hidden=hparams['hidden'], 
    dropout=hparams['dropout']
).to(device)

optimizer = torch.optim.Adam(
    final_model.parameters(), 
    lr=hparams['lr'], 
    weight_decay=hparams['weight_decay']
)
criterion = nn.CrossEntropyLoss()

# 3. Bucle de entrenamiento final
EPOCHS = hparams.get('epochs', 100) # Usar epochs del HPO

for epoch in range(1, EPOCHS + 1):
    final_model.train()
    optimizer.zero_grad()
    
    logits = final_model(X_scaled_t)
    loss = criterion(logits, y_full_t)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0 or epoch == EPOCHS:
        # Calcular accuracy en el set de entrenamiento (solo como referencia)
        final_model.eval()
        with torch.inference_mode():
            train_acc = (logits.argmax(dim=1) == y_full_t).float().mean().item()
        print(f"Epoch {epoch}/{EPOCHS} | Loss: {loss.item():.4f} | Train Accuracy: {train_acc:.4f}")

# 4. Guardar el estado del modelo (state_dict)
model_path = ITELConfig.MODELS_DIR / 'final_mlp_model.pt'
torch.save(final_model.state_dict(), model_path)

print(f"\n✅ Modelo MLP final entrenado y guardado en: {model_path}")

Entrenando el Modelo MLP final sobre TODOS los datos...
Epoch 10/100 | Loss: 0.3635 | Train Accuracy: 0.8416
Epoch 20/100 | Loss: 0.2542 | Train Accuracy: 0.8973
Epoch 30/100 | Loss: 0.1804 | Train Accuracy: 0.9287
Epoch 40/100 | Loss: 0.1153 | Train Accuracy: 0.9563
Epoch 50/100 | Loss: 0.0742 | Train Accuracy: 0.9743
Epoch 60/100 | Loss: 0.0512 | Train Accuracy: 0.9812
Epoch 70/100 | Loss: 0.0366 | Train Accuracy: 0.9869
Epoch 80/100 | Loss: 0.0232 | Train Accuracy: 0.9925
Epoch 90/100 | Loss: 0.0142 | Train Accuracy: 0.9962
Epoch 100/100 | Loss: 0.0118 | Train Accuracy: 0.9960

✅ Modelo MLP final entrenado y guardado en: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos\final_mlp_model.pt


In [8]:
# 8.6 - Exportar el Modelo Final a ONNX (Opcional pero recomendado)

print("Exportando modelo final a formato ONNX...")

try:
    # 1. Asegurarse de que el modelo esté en modo evaluación
    final_model.eval()

    # 2. Crear una entrada de ejemplo (dummy input)
    # Necesita tener el shape correcto: (1, 4608)
    dummy_input = X_scaled_t[0].unsqueeze(0).to(device) 
    
    onnx_path = ITELConfig.MODELS_DIR / 'final_mlp_model.onnx'

    # 3. Exportar
    torch.onnx.export(
        final_model,               # El modelo a exportar
        dummy_input,             # Una entrada de ejemplo
        onnx_path,               # Dónde guardar el archivo
        input_names=['input_embedding'], # Nombre de la entrada
        output_names=['output_logits'], # Nombre de la salida
        dynamic_axes={'input_embedding': {0: 'batch_size'}, # Permitir batch_size variable
                      'output_logits': {0: 'batch_size'}},
        opset_version=11 # Versión de ONNX
    )
    
    print(f"✅ Modelo exportado a ONNX exitosamente en: {onnx_path}")
    print("Esta versión .onnx es ideal para servidores de inferencia optimizados (como ONNX Runtime).")

except Exception as e:
    print(f"⚠️ Ocurrió un error al exportar a ONNX: {e}")

Exportando modelo final a formato ONNX...


C:\Users\USER\AppData\Local\Temp\ipykernel_21960\2036737191.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


✅ Modelo exportado a ONNX exitosamente en: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos\final_mlp_model.onnx
Esta versión .onnx es ideal para servidores de inferencia optimizados (como ONNX Runtime).


# 8.7 - Resumen de Exportación

¡Exportación completada!

Para usar tu modelo en una página web (mediante una API de Python como Flask o FastAPI), necesitarás los siguientes componentes en tu servidor:

1.  **El Modelo:** `Modelos/final_mlp_model.pt` (para PyTorch) o `Modelos/final_mlp_model.onnx` (para ONNX Runtime).
2.  **El Scaler:** `Modelos/final_mlp_scaler.joblib` (para Scikit-learn).
3.  **Las Funciones:** Las definiciones de `procesar_audio()` (de 04) y `extraer_embedding()` (de 05), junto con sus dependencias (HuBERT, librosa, etc.).

Tu pipeline de inferencia en el servidor será:
**WAV (Usuario)** -> `procesar_audio()` -> `extraer_embedding()` -> `final_scaler.transform()` -> `final_model.forward()` -> **JSON (Resultado)**